# NB_patient_360

Builds `dbx_caresync.gold.patient_360` — one row per patient enriched with hospital name, insurance provider name, and aggregated lab-test metrics.

In [0]:
%sql
CREATE OR REPLACE TABLE dbx_caresync.gold.patient_360
AS
WITH lab_agg AS (
  SELECT
    patient_id,
    COUNT(*)                                                        AS total_tests,
    COUNT(CASE WHEN result_status = 'Abnormal' THEN 1 END)         AS abnormal_test_count,
    COUNT(CASE WHEN result_status = 'Critical' THEN 1 END)         AS critical_test_count,
    MAX(test_date)                                                  AS last_test_date
  FROM dbx_caresync.silver.lab_results
  GROUP BY patient_id
)
SELECT
  -- === Patient Core ===
  p.patient_id,
  p.first_name,
  p.last_name,
  CAST(p.date_of_birth AS DATE)                                     AS date_of_birth,
  FLOOR(DATEDIFF(CURRENT_DATE(), CAST(p.date_of_birth AS DATE)) / 365.25)  AS age,
  p.gender,
  p.city,
  p.state,

  -- === Hospital ===
  p.registered_hospital_id                                          AS hospital_id,
  h.hospital_name,

  -- === Insurance ===
  p.insurance_provider_id,
  ip.provider_name                                                  AS insurance_provider_name,

  -- === Lab Aggregates ===
  COALESCE(la.total_tests, 0)                                       AS total_tests,
  COALESCE(la.abnormal_test_count, 0)                               AS abnormal_test_count,
  COALESCE(la.critical_test_count, 0)                               AS critical_test_count,
  la.last_test_date

FROM dbx_caresync.silver.patients                     AS p
LEFT JOIN dbx_caresync.silver.hospitals               AS h
       ON p.registered_hospital_id = h.hospital_id
LEFT JOIN dbx_caresync.silver.insurance_providers     AS ip
       ON p.insurance_provider_id = ip.insurance_provider_id
LEFT JOIN lab_agg                                     AS la
       ON p.patient_id = la.patient_id
ORDER BY p.patient_id;

In [0]:
updated_df = spark.table("dbx_caresync.gold.patient_360")
records_count = updated_df.count()
dbutils.notebook.exit(str(records_count))